In [1]:
import numpy as np, pandas as pd
from datetime import datetime
from typing import Any, Dict
from pydantic import BaseModel, Field

class Event(BaseModel):
    event_id: int
    timestamp: datetime                 # when it happened
    entity: str                         # who/what it is about (here: a host)
    source_model: str                   # which detector raised it
    threat_type: str                    # BruteForce, PortScan, InsiderMisuse, Ransomware, ...
    raw_score: float = Field(ge=0, le=1)   # detector confidence, normalised to 0-1
    technique: str                      # MITRE ATT&CK ID (verify each on attack.mitre.org)
    label: int = 0                      # ground truth: used ONLY to evaluate, never to score
    context: Dict[str, Any] = {}        # filled by the context engine

In [2]:
rng = np.random.default_rng(11)
T0 = pd.Timestamp("2026-01-01")        # simulation start; day d = T0 + d days
N_HOSTS = 150

net = pd.read_csv("../data/processed/network_scores.csv")
ins = pd.read_csv("../data/processed/insider_scores.csv")
ran = pd.read_csv("../data/processed/ransomware_scores.csv")

ins["pct"] = ins["score"].rank(pct=True)           # convert anomaly score to 0-1 percentile
net_a = net[net["score"] >= 0.5].reset_index(drop=True)
ins_a = ins[ins["pct"] >= 0.95].copy()
ran_a = ran[ran["score"] >= 0.5].copy()
print("raw alerts -> network:", len(net_a), "| insider:", len(ins_a), "| ransomware:", len(ran_a))

bad = ins[ins["label"] == 1].groupby("user")["day"].min()   # user -> first malicious day
insider_users = list(bad.index)                              # the 8 insiders (used for planting)

raw alerts -> network: 127322 | insider: 226 | ransomware: 1275


In [7]:
ins_a["pct"] = (ins_a["pct"] - 0.95) / 0.05     # 0 = barely in the top 5%, 1 = most extreme

In [8]:
# ---- insider events: user u uses host_u; time = that day + random second ----
ev_ins = pd.DataFrame({
    "timestamp": T0 + pd.to_timedelta(ins_a["day"].values, unit="D")
                    + pd.to_timedelta(rng.integers(0, 86400, len(ins_a)), unit="s"),
    "entity": ("host_" + ins_a["user"].astype(str)).values,
    "source_model": "insider_iforest", "threat_type": "InsiderMisuse",
    "raw_score": ins_a["pct"].values, "label": ins_a["label"].values})

# ---- ransomware events: each run is assigned a host and a day ----
true_runs = ran_a.loc[ran_a["label"] == 1, "run_id"].unique()
run_host, run_day, run_off = {}, {}, {}
for r in ran_a["run_id"].unique():
    run_host[r] = int(rng.integers(0, N_HOSTS)); run_day[r] = int(rng.integers(60, 90))
    run_off[r] = int(rng.integers(0, 80000))
for r, u in zip(true_runs[:4], insider_users[:4]):    # PLANTED: 4 ransomware runs hit insider hosts the day after
    run_host[r] = u; run_day[r] = int(bad[u]) + 1
ev_ran = pd.DataFrame({
    "timestamp": [T0 + pd.Timedelta(days=run_day[r], seconds=run_off[r] + 10 * int(w))
                  for r, w in zip(ran_a["run_id"], ran_a["window"])],
    "entity": ["host_%d" % run_host[r] for r in ran_a["run_id"]],
    "source_model": "ransomware_xgb", "threat_type": "Ransomware",
    "raw_score": ran_a["score"].values, "label": ran_a["label"].values})

# ---- network events: group flows into 'campaigns' of up to 500 flows on one host ----
parts, plant_i = [], 0
for fam, g in net_a.groupby("family"):
    idx = g.index.to_numpy()
    for s in range(0, len(idx), 500):
        chunk = idx[s:s + 500]
        if fam == "PortScan" and plant_i < len(insider_users):   # PLANTED: recon 2 days before an insider turns bad
            host = insider_users[plant_i]; day = max(int(bad[host]) - 2, 60); plant_i += 1
        else:
            host = int(rng.integers(0, N_HOSTS)); day = int(rng.integers(60, 90))
        t0 = T0 + pd.Timedelta(days=day, seconds=int(rng.integers(0, 80000)))
        parts.append(pd.DataFrame({"i": chunk,
                                   "timestamp": t0 + pd.to_timedelta(np.arange(len(chunk)) * 0.5, unit="s"),
                                   "entity": f"host_{host}"}))
st = pd.concat(parts).set_index("i")
n = net_a.join(st)
ev_net = pd.DataFrame({
    "timestamp": n["timestamp"], "entity": n["entity"], "source_model": "network_xgb",
    "threat_type": np.where(n["family"] == "BENIGN", "NetworkAnomaly", n["family"]),
    "raw_score": n["score"], "label": n["label"]})

ev = pd.concat([ev_net, ev_ins, ev_ran], ignore_index=True)
ev["event_id"] = np.arange(len(ev))
TECH = {"BruteForce": "T1110", "PortScan": "T1046", "DoS": "T1498", "DDoS": "T1498",
        "WebAttack": "T1190", "Bot": "T1071", "InsiderMisuse": "T1078",
        "Ransomware": "T1486", "NetworkAnomaly": "unmapped"}      # verify these on attack.mitre.org
ev["technique"] = ev["threat_type"].map(TECH)
STAGE = {"PortScan": 1, "BruteForce": 2, "WebAttack": 2, "Bot": 2,
         "InsiderMisuse": 3, "Ransomware": 4, "DoS": 4, "DDoS": 4, "NetworkAnomaly": 0}   # rough kill-chain order

# validate a sample against the schema (in production every event would pass through it)
for r in ev.sample(2000, random_state=1).to_dict("records"):
    Event(**{k: r[k] for k in ["event_id", "timestamp", "entity", "source_model",
                               "threat_type", "raw_score", "technique", "label"]})
print("schema OK;", len(ev), "events")

schema OK; 128823 events


In [9]:
SENSITIVE = {f"host_{i}" for i in range(15)}           # pretend hosts 0-14 are servers holding sensitive data
ev["off_hours"] = (ev["timestamp"].dt.hour < 7) | (ev["timestamp"].dt.hour >= 20)
ev["sensitive"] = ev["entity"].isin(SENSITIVE)
ev["authorized"] = True                                 # IAM says "permitted" for all of them

# multipliers are my assumptions; justify or tune them on validation data
mult = 1 + 0.20 * ev["off_hours"] + 0.30 * ev["sensitive"]
s = ev["raw_score"].clip(0.01, 0.99)       # avoid dividing by zero at exactly 0 or 1
odds = s / (1 - s) * mult                  # context scales the ODDS (evidence strength)
ev["score"] = odds / (1 + odds)            # convert back to a 0-1 score, never exactly 1

In [10]:
GAP = pd.Timedelta(hours=24)
ev = ev.sort_values(["entity", "timestamp"]).reset_index(drop=True)
gap = ev.groupby("entity")["timestamp"].diff()           # time since the previous event on the same host
new_incident = ev["entity"].ne(ev["entity"].shift()) | (gap > GAP)
ev["incident_id"] = new_incident.cumsum()

def score_incident(g):
    best = g.groupby("threat_type")["score"].max()                    # strongest evidence per threat type
    base = 1 - np.prod(1 - best.values)                               # noisy-OR across different types
    stages = g["threat_type"].map(STAGE)
    n_stages = stages[stages > 0].nunique()                           # distinct kill-chain stages
    # each extra kill-chain stage shrinks the "remaining doubt" by 30%; result always stays below 1
    incident_score = 1 - (1 - base) * (0.7 ** max(n_stages - 1, 0))
    return pd.Series({"entity": g["entity"].iloc[0], "n_events": len(g), "n_types": best.size,
                      "n_stages": n_stages, "incident_score": incident_score,
                      "true": int(g["label"].max())})            # lenient: true if ANY member is a real attack

inc = ev.groupby("incident_id").apply(score_incident)

In [11]:
from sklearn.metrics import average_precision_score
print("raw alerts:", len(ev), "-> incidents:", len(inc),
      f"({1 - len(inc)/len(ev):.1%} reduction)")
print(ev.groupby("source_model").size())                              # where the raw alerts come from
print("false alerts:", int((ev['label'] == 0).sum()), "-> false incidents:", int((inc['true'] == 0).sum()))
multi = inc[inc["n_stages"] >= 2]
print("multi-stage incidents:", len(multi), "| fraction truly malicious:", round(multi["true"].mean(), 3))
print("incident-score PR-AUC:", round(average_precision_score(inc["true"], inc["incident_score"]), 3),
      "| baseline:", round(inc["true"].mean(), 3))
print(inc.sort_values("incident_score", ascending=False).head(10))

ev.to_csv("../data/processed/events.csv", index=False)
inc.to_csv("../data/processed/incidents.csv")  # the dashboard reads these in Phase 7

# 1) reduction WITHOUT the flow floods, which is a fairer view
non_net = ev[ev["source_model"] != "network_xgb"]
print("insider+ransomware alerts:", len(non_net), "-> incidents they belong to:", non_net["incident_id"].nunique())

# 2) can an analyst find real incidents first? precision in the top 20 and top 50
for k in [20, 50]:
    print(f"precision in top {k} incidents:", round(inc.nlargest(k, "incident_score")["true"].mean(), 3))

# 3) how many distinct score values do we have (1.0 everywhere = broken)
print("unique incident scores:", inc["incident_score"].round(4).nunique())

raw alerts: 128823 -> incidents: 426 (99.7% reduction)
source_model
insider_iforest       226
network_xgb        127322
ransomware_xgb       1275
dtype: int64
false alerts: 157 -> false incidents: 103
multi-stage incidents: 29 | fraction truly malicious: 1.0
incident-score PR-AUC: 0.995 | baseline: 0.758
               entity  n_events  n_types  n_stages  incident_score  true
incident_id                                                             
224           host_35      1535        3         3        0.999998     1
213           host_32      1001        3         3        0.999979     1
3              host_0      1000        2         2        0.999964     1
70           host_121      1000        2         2        0.999951     1
389           host_92       646        2         2        0.999942     1
129          host_142      1000        2         2        0.999942     1
393           host_94      1000        2         2        0.999942     1
333           host_75       522      

In [12]:
# ranking WITHOUT correlation logic: just the strongest raw detector score in each incident
max_raw = ev.groupby("incident_id")["raw_score"].max()
inc["max_raw"] = max_raw

for name, col in [("max raw score only", "max_raw"), ("full incident score", "incident_score")]:
    pr = average_precision_score(inc["true"], inc[col])
    top50 = inc.nlargest(50, col)["true"].mean()
    print(f"{name:20s} PR-AUC={pr:.3f}  precision@50={top50:.3f}")

max raw score only   PR-AUC=0.995  precision@50=1.000
full incident score  PR-AUC=0.995  precision@50=1.000
